In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import cross_val_score, KFold
from sklearn.preprocessing import RobustScaler, LabelEncoder
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import Lasso, ElasticNet
from sklearn.ensemble import GradientBoostingRegressor
from mlxtend.regressor import StackingCVRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
import scipy.stats as stats

# 1. 加载数据
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

# 2. 数据预处理
# 剔除离群点 (根据 GrLivArea 散点图)
train = train.drop(train[(train['GrLivArea']>4000) & (train['SalePrice']<300000)].index)

# 对目标变量进行对数转换 (让分布更趋于正态)
y = np.log1p(train['SalePrice'])
train_id = train['Id']
test_id = test['Id']
all_data = pd.concat((train, test)).drop(['Id', 'SalePrice'], axis=1)

# 3. 特征工程
# 处理缺失值：根据业务逻辑填补
none_cols = ['PoolQC', 'MiscFeature', 'Alley', 'Fence', 'FireplaceQu', 'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'MasVnrType']
for col in none_cols:
    all_data[col] = all_data[col].fillna('None')

zero_cols = ['GarageYrBlt', 'GarageArea', 'GarageCars', 'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', 'BsmtFullBath', 'BsmtHalfBath', 'MasVnrArea']
for col in zero_cols:
    all_data[col] = all_data[col].fillna(0)

# LotFrontage 按邻居中位数填充
all_data["LotFrontage"] = all_data.groupby("Neighborhood")["LotFrontage"].transform(lambda x: x.fillna(x.median()))

# 构造关键特征：总面积
all_data['TotalSF'] = all_data['TotalBsmtSF'] + all_data['1stFlrSF'] + all_data['2ndFlrSF']

# 类别转换：将数值型的类别转为字符串
all_data['MSSubClass'] = all_data['MSSubClass'].apply(str)
all_data['YrSold'] = all_data['YrSold'].astype(str)
all_data['MoSold'] = all_data['MoSold'].astype(str)

# 独热编码
all_data = pd.get_dummies(all_data)

# 拆分训练集和测试集
X = all_data[:len(y)]
test_final = all_data[len(y):]

# 4. 模型构建 (Stacking 策略)
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# 定义基础模型
lasso = make_pipeline(RobustScaler(), Lasso(alpha=0.0005, random_state=1))
enet = make_pipeline(RobustScaler(), ElasticNet(alpha=0.0005, l1_ratio=.9, random_state=3))
gbr = GradientBoostingRegressor(n_estimators=3000, learning_rate=0.05, max_depth=4, random_state=5)
lightgbm = LGBMRegressor(objective='regression', num_leaves=5, learning_rate=0.05, n_estimators=720)
xgboost = XGBRegressor(learning_rate=0.05, n_estimators=3000, max_depth=3, subsample=0.7, colsample_bytree=0.7)

# 使用 Stacking 进行融合
stack_gen = StackingCVRegressor(regressors=(lasso, enet, gbr, xgboost, lightgbm),
                                meta_regressor=xgboost,
                                use_features_in_secondary=True)

# 5. 训练与预测
print("正在拟合 Stacking 模型...")
stack_gen_model = stack_gen.fit(np.array(X), np.array(y))

# 最终预测结果 (指数还原)
final_predictions = np.expm1(stack_gen_model.predict(np.array(test_final)))

# 保存结果
submission = pd.DataFrame({'Id': test_id, 'SalePrice': final_predictions})
submission.to_csv('submission.csv', index=False)
print("预测完成，结果已保存为 submission.csv")
